In [11]:
!pip install prophet

In [12]:
from prophet import Prophet
print("Prophet installed successfully")


Prophet installed successfully


In [13]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
from statsmodels.tsa.seasonal import seasonal_decompose
from prophet import Prophet
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import warnings
import os

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-darkgrid')

# Create directories for deliverables
os.makedirs('visualizations', exist_ok=True)
os.makedirs('powerbi_data', exist_ok=True)  # Changed folder name for clarity

print(" STARTING PROJECT 2: SALES FORECASTING PIPELINE")


 STARTING PROJECT 2: SALES FORECASTING PIPELINE


In [14]:
# 1. DATA LOADING & MERGING
try:
    # Load core datasets
    df_train = pd.read_csv('train.csv')
    df_stores = pd.read_csv('stores.csv')
    df_holidays = pd.read_csv('holidays_events.csv')
    
    # Merge store metadata for "Region-wise insights"
    df_merged = df_train.merge(df_stores, on='store_nbr', how='left')
    df_merged['date'] = pd.to_datetime(df_merged['date'])
    
    print(f"✓ Data Loaded. Full shape: {df_merged.shape}")
    print(f"✓ Date Range: {df_merged['date'].min()} to {df_merged['date'].max()}")

except FileNotFoundError:
    print("ERROR: Data files not found. Please upload 'train.csv' and 'stores.csv'.")
    raise

✓ Data Loaded. Full shape: (3000888, 10)
✓ Date Range: 2013-01-01 00:00:00 to 2017-08-15 00:00:00


In [15]:
# Data Cleaning
# Handle missing values
df_merged['sales'] = df_merged['sales'].fillna(0)
df_merged['onpromotion'] = df_merged['onpromotion'].fillna(0)

# Remove outliers/negatives
df_merged = df_merged[df_merged['sales'] >= 0]

print("✓ Data cleaning complete.")

✓ Data cleaning complete.


In [16]:
# DETAILED EDA & BUSINESS INSIGHTS
# Insight A: Product Category Performance
category_sales = df_merged.groupby('family')['sales'].sum().sort_values(ascending=False)
top_5_cat = category_sales.head(5)
bottom_5_cat = category_sales.tail(5)

print("\n--- Top 5 Performing Categories ---")
print(top_5_cat.to_string())
print("\n--- Bottom 5 (Underperforming) Categories ---")
print(bottom_5_cat.to_string())

# Plot Category Distribution (Saved for Report)
plt.figure(figsize=(12, 6))
sns.barplot(x=top_5_cat.values, y=top_5_cat.index, palette='viridis')
plt.title('Top 5 Product Categories by Revenue')
plt.xlabel('Total Sales')
plt.tight_layout()
plt.savefig('visualizations/01_top_categories.png')
plt.close()

# Insight B: Region-wise Insights
# We look at sales by State/City from the stores data
region_sales = df_merged.groupby('state')['sales'].sum().sort_values(ascending=False).head(10)

plt.figure(figsize=(12, 6))
sns.barplot(x=region_sales.values, y=region_sales.index, palette='magma')
plt.title('Top 10 States by Sales Volume')
plt.xlabel('Total Sales')
plt.tight_layout()
plt.savefig('visualizations/02_regional_sales.png')
plt.close()

# --- EXPORT FOR POWER BI (PART 1) ---
# We save this granular data for slicers (State, City, Category)
dashboard_kpis = df_merged.groupby(['date', 'family', 'state', 'city', 'store_nbr'])['sales'].sum().reset_index()
dashboard_kpis.to_csv('powerbi_data/historical_sales_granular.csv', index=False)
print("✓ Granular history exported for Power BI (historical_sales_granular.csv).")


--- Top 5 Performing Categories ---
family
GROCERY I    3.434627e+08
BEVERAGES    2.169545e+08
PRODUCE      1.227047e+08
CLEANING     9.752129e+07
DAIRY        6.448771e+07

--- Bottom 5 (Underperforming) Categories ---
family
MAGAZINES          266359.0
HARDWARE           103470.0
HOME APPLIANCES     41601.0
BABY CARE           10051.0
BOOKS                6438.0
✓ Granular history exported for Power BI (historical_sales_granular.csv).


In [17]:
# 4. PREPARING FOR TIME SERIES MODELING
# -------------------------------------------------------------------------
# Now we aggregate to Daily level for the Forecasting Model
df_daily = df_merged.groupby('date').agg({
    'sales': 'sum',
    'onpromotion': 'sum'
}).reset_index()

# Feature Engineering
df_daily['dayofweek'] = df_daily['date'].dt.dayofweek
df_daily['month'] = df_daily['date'].dt.month
df_daily['day'] = df_daily['date'].dt.day
df_daily['is_weekend'] = df_daily['dayofweek'].isin([5, 6]).astype(int)

# Lag Features for Random Forest
df_daily['sales_lag_7'] = df_daily['sales'].shift(7)
df_daily['sales_lag_30'] = df_daily['sales'].shift(30)
df_daily = df_daily.dropna() # Drop rows with NaNs from lags

# Split Train/Test
train_size = int(len(df_daily) * 0.9)
train_set = df_daily.iloc[:train_size]
test_set = df_daily.iloc[train_size:]

print(f"\n✓ Modeling Data Prepared. Train size: {len(train_set)}, Test size: {len(test_set)}")


✓ Modeling Data Prepared. Train size: 1488, Test size: 166


In [18]:
# 5. MODELING: PROPHET vs RANDOM FOREST
# -------------------------------------------------------------------------

# --- Model A: Prophet ---
print("\n Training Prophet Model...")
prophet_df = train_set[['date', 'sales']].rename(columns={'date': 'ds', 'sales': 'y'})
m_prophet = Prophet(daily_seasonality=True, yearly_seasonality=True)
m_prophet.fit(prophet_df)

# Predict on Test
future_prophet = test_set[['date']].rename(columns={'date': 'ds'})
forecast_prophet = m_prophet.predict(future_prophet)
preds_prophet = forecast_prophet['yhat'].values

# --- Model B: Random Forest ---
print(" Training Random Forest...")
features = ['dayofweek', 'month', 'day', 'is_weekend', 'onpromotion', 'sales_lag_7', 'sales_lag_30']
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(train_set[features], train_set['sales'])

# Predict on Test
preds_rf = rf.predict(test_set[features])

16:14:38 - cmdstanpy - INFO - Chain [1] start processing



 Training Prophet Model...


16:14:39 - cmdstanpy - INFO - Chain [1] done processing


 Training Random Forest...


In [19]:
# 6. EVALUATION
# -------------------------------------------------------------------------
def get_metrics(y_true, y_pred, name):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    return {'Model': name, 'MAE': mae, 'RMSE': rmse, 'MAPE (%)': mape}

metrics_p = get_metrics(test_set['sales'], preds_prophet, 'Prophet')
metrics_rf = get_metrics(test_set['sales'], preds_rf, 'Random Forest')

results_df = pd.DataFrame([metrics_p, metrics_rf])
print("\n MODEL EVALUATION RESULTS:")
print(results_df)

best_model_name = results_df.sort_values('MAPE (%)').iloc[0]['Model']
print(f"\n Best Model Selected: {best_model_name}")


 MODEL EVALUATION RESULTS:
           Model           MAE           RMSE  MAPE (%)
0        Prophet  73112.778260  109034.059379  7.755442
1  Random Forest  60551.287896   85936.300587  6.665329

 Best Model Selected: Random Forest


In [21]:
# 7. FUTURE FORECAST & RECOMMENDATIONS
# -------------------------------------------------------------------------
print(f"\n Generating 30-Day Forecast using {best_model_name}...")

last_date = df_daily['date'].max()
future_dates = pd.date_range(last_date + timedelta(days=1), periods=30)

if best_model_name == 'Prophet':
    future_df = pd.DataFrame({'ds': future_dates})
    forecast = m_prophet.predict(future_df)
    predictions = forecast['yhat'].values
else:
    # Logic for RF future prediction (using averages for simplicity in demo)
    future_data = pd.DataFrame({'date': future_dates})
    future_data['dayofweek'] = future_data['date'].dt.dayofweek
    future_data['month'] = future_data['date'].dt.month
    future_data['day'] = future_data['date'].dt.day
    future_data['is_weekend'] = future_data['dayofweek'].isin([5,6]).astype(int)
    future_data['onpromotion'] = df_daily['onpromotion'].mean()
    future_data['sales_lag_7'] = df_daily['sales'].iloc[-7:].mean()
    future_data['sales_lag_30'] = df_daily['sales'].iloc[-30:].mean()
    predictions = rf.predict(future_data[features])


 Generating 30-Day Forecast using Random Forest...


In [23]:
# Create forecast dataframe
final_forecast = pd.DataFrame({
    'date': future_dates,
    'predicted_sales': predictions,
    'type': 'Forecast'
})

# Combine history and forecast for seamless plotting
history_export = df_daily[['date', 'sales']].copy()
history_export['type'] = 'Historical'
history_export.rename(columns={'sales': 'predicted_sales'}, inplace=True)

full_export = pd.concat([history_export, final_forecast])
full_export.to_csv('powerbi_data/final_forecast_for_powerbi.csv', index=False)